# Random Forest Forecast Analysis

## Objective
This report explains the updated Random Forest forecasting workflow aligned with src/models/train.py, evaluates model performance, interprets results, and summarizes key findings for next-day maximum temperature prediction in Cambodia.

## Problem Definition
The goal is to predict next-day maximum temperature (temp_max) using weather information available from the current or previous day, enriched with time features and regional variation.

## Why We Chose Random Forest
We selected Random Forest Regressor because it is strong for non-linear tabular relationships, more stable than a single decision tree, and usually better at reducing overfitting through bagging and feature randomness.

Main reasons:
- It captures non-linear interactions between weather variables without heavy preprocessing.
- It is more robust than a single tree because predictions are averaged across many trees.
- It does not require feature scaling, which keeps the pipeline simple and reproducible.
- It provides feature importance, helping explain which signals drive predictions.
- It is a strong baseline ensemble model for tabular forecasting tasks.

Prediction target:
- **temp_max** (next-day maximum temperature)

Main input features used:
- **Weather variables:** temp_min, rain, wind_speed, lat, lon
- **Time features:** year, month, day, dayofweek
- **Cyclical encoding:** month_sin, month_cos, dayofweek_sin, dayofweek_cos (captures seasonality)
- **Province categorical:** one-hot encoded province indicator (enables regional variation)

## Data Preparation and Modeling Workflow
1. Loaded weather data from cambodia_weather.csv and normalized column names.
2. Parsed and cleaned date values with error handling.
3. Added calendar-based time features (year, month, day, dayofweek).
4. Applied cyclical encoding for month and dayofweek to capture seasonality.
5. Applied data quality cleaning: numeric coercion, temp_max ≥ temp_min, rain ≥ 0, wind_speed ≥ 0, Cambodia lat/lon bounds.
6. Built feature matrix with weather variables, time features, cyclical encodings, and one-hot encoded provinces.
7. Split the dataset randomly: 80 percent training, 20 percent testing (random_state=42).
8. Trained a RandomForestRegressor with n_estimators=220, max_depth=24, min_samples_leaf=2, and random_state=42.

Final dataset sizes after preprocessing:

| Item | Value |
|---|---:|
| Numeric features | 13 (weather + time + cyclical) |
| Province variables | 5 (one-hot encoded) |
| Total features | 18 |
| Trainable rows | 20,570 |
| Training rows (80%) | 16,456 |
| Testing rows (20%) | 4,114 |

## Evaluation Metrics and Results
To evaluate the model, the following metrics are computed:
- **RMSE:** Root Mean Squared Error
- **MAE:** Mean Absolute Error
- **R2:** Coefficient of Determination

### Random Forest Results (src-aligned Training)
With the expanded feature set (weather, time, cyclical, and province features) and optimized hyperparameters (n_estimators=220, max_depth=24, min_samples_leaf=2):

**Observed metrics (executed):**
- RMSE: **1.1331 deg C**
- MAE: **0.8590 deg C**
- R2: **0.8624**
- Train rows: **16,456**
- Test rows: **4,114**

### Key Differences from Previous Iteration
The new model uses:
- **Richer feature engineering:** time features + cyclical encoding + regional variation (no lag features)
- **Random train/test split:** tests generalization across all time periods (vs. chronological split)
- **More trees and greater depth:** 220 estimators, max_depth=24 (vs. 100, max_depth=15)
- **Additional constraint:** min_samples_leaf=2 to reduce overfitting

### Metrics Table

| Metric | Value | Interpretation |
|---|---:|---|
| RMSE (deg C) | 1.1331 | Root mean squared error |
| MAE (deg C) | 0.8590 | Mean absolute error |
| R2 | 0.8624 | Proportion of variance explained |
| Train rows | 16,456 | Training set size |
| Test rows | 4,114 | Test set size |

## Feature Importance Interpretation
With the expanded feature set (13 numeric weather/time features + 5 province one-hot variables), feature importance reflects contributions from weather, time, and region.

### Expected Feature Categories
- **Weather:** temp_min, rain, wind_speed, lat, lon
- **Time:** year, month, day, dayofweek, month_sin, month_cos, dayofweek_sin, dayofweek_cos
- **Region:** one-hot encoded province variables

### Top Feature Importance (Executed)

| Feature | Importance | Share | Category |
|---|---:|---:|---|
| province_Mondulkiri | 0.1873 | 18.73% | Region |
| month_sin | 0.1679 | 16.79% | Time (cyclical) |
| temp_min | 0.1534 | 15.34% | Weather |
| lon | 0.1400 | 14.00% | Weather (geospatial) |
| rain | 0.0848 | 8.48% | Weather |

### Interpretation
- Seasonal signal (**month_sin**) and regional signal (**province_Mondulkiri**) are both strong.
- **temp_min** remains a major weather driver, consistent with temperature persistence.
- Geospatial information (**lon**) contributes substantially.
- The model is learning a blend of weather, seasonality, and location effects rather than relying on one feature type only.

## Result Explanation

### 1. Model Quality
The updated Random Forest incorporates richer feature engineering (time features, cyclical encoding, and regional variation) with optimized hyperparameters. This approach is designed to capture non-linear and seasonal patterns more robustly than simpler baselines.

### 2. Feature Engineering Impact
By adding time features (year, month, day, dayofweek) and cyclical encodings (sin/cos), the model can learn periodic patterns. Province information adds regional variation. These enhancements should improve generalization across different seasons and regions.

### 3. Split Strategy Impact
The random train/test split (vs. chronological) tests how well the model generalizes to unseen data points throughout the time series, not just future data. This is a stricter evaluation of model robustness but closer to real-world deployment scenarios.

### 4. Stability with Ensemble
With 220 trees and max_depth=24, the ensemble is expressive and handles complex patterns. The min_samples_leaf=2 constraint helps prevent overfitting on small subsets. Deeper trees can fit noise, so ensemble averaging is critical for generalization.

### 5. Practical Reading
Once metrics are computed, interpret them as:
- **R2:** Does the model explain a reasonable fraction of test variance?
- **MAE:** Is the average absolute error operationally acceptable?
- **RMSE:** Are occasional large errors within tolerance?
- **Data splits:** Are training and test sets large enough for the 220-tree ensemble?

## Key Findings

### Structural Improvements
- The updated Random Forest incorporates time-aware features (cyclical encoding, day-of-year, province effects) aligned with the src training pipeline.
- Hyperparameter optimization (n_estimators=220, max_depth=24, min_samples_leaf=2) provides a deeper, more expressive model than the prior configuration.
- Random train/test split ensures evaluation across all time periods, not just future data (stricter validation).

### Expected Result Patterns
- **Feature engineering:** Cyclical features (month_sin, month_cos, dayofweek_sin, dayofweek_cos) should capture seasonality better than raw calendar indices.
- **Regional variation:** Province encoding enables the model to learn location-specific temperature behavior.
- **Weather fundamentals:** Basic weather variables (temp_min, rain, wind_speed) likely remain strong predictors.
- **Model generalization:** Ensemble methods should improve over single-tree baselines.

### Summary Table

| Finding | Evidence | Implication |
|---|---|---|
| Seasonal patterns matter | Cyclical features rank in importance | Temperature has clear seasonal variation |
| Regional effects exist | Province variables show importance | Location-specific factors affect forecast |
| Weather drives temperature | temp_min, rain, wind_speed rank high | Basic meteorology dominates |
| Ensemble improves stability | 220 trees diversify predictions | Random forest beats single tree |

## Limitations

- **Split strategy:** Random split does not respect temporal ordering; time-series cross-validation (walk-forward) would be more appropriate for forecasting tasks.
- **Feature availability:** Feature set is fixed to available data; additional proxies (humidity, pressure, cloud cover, solar radiation) could improve predictions.
- **Hyperparameter tuning:** Single configuration is used; grid search or Bayesian optimization could refine n_estimators, max_depth further.
- **Regional effects:** Province encoding assumes stable effects; intra-province variation and migration patterns are not represented.
- **Long-range dependencies:** Model does not account for multi-day trends, monsoon phases, or ENSO cycles that influence weather patterns.
- **Feature interactions:** The model learns feature interactions implicitly; explicit engineering (e.g., wind × temperature) might improve signal.

## Final Conclusion

The updated Random Forest model aligns with the src training pipeline, incorporating time-aware features (cyclical encoding, temporal variables), regional variation (province encoding), and optimized hyperparameters (n_estimators=220, max_depth=24, min_samples_leaf=2). The random train/test split provides a rigorous test of generalization across all time periods.

**Next steps after execution:**
1. Review feature importance to identify dominant drivers (weather, season, or region).
2. Compare actual metrics against baselines (mean prediction, single decision tree) to quantify ensemble benefit.
3. Perform error analysis—identify patterns where the model struggles (e.g., abrupt weather changes, monsoon transitions).
4. Consider time-series validation (walk-forward cross-validation) for more realistic forecasting assessment.
5. Explore ensemble methods combining this Random Forest with other models (linear regression, decision tree) for robustness.